# TenVoices · wav2vec 2.0 on a Colab GPU

Runs the shared training script `support/train_wav2vec2.py` on a Colab GPU. Every run is just a set of options; this notebook adds no logic of its own.

| Run (task) | `EPOCHS` | `NOISE` | `UNTRAINED` |
|---|---|---|---|
| 15 epochs + white noise (T15, M1) | 15 | `white` | `False` |
| 8 epochs + white noise (T16, M2) | 8 | `white` | `False` |
| 3 epochs (T17, M3) | 3 | `none` | `False` |
| Untrained baseline (T18, M4) | – | – | `True` |
| 15 epochs, no noise (T19, M1) | 15 | `none` | `False` |

**Before you start:** *Runtime → Change runtime type → T4 GPU*. The original dataset archives must be in the team Drive under `TenVoices/raw/` (see `data/SOURCES.md`), and the subsets and splits (T42, T08) must be merged into `main`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Settings

In [ ]:
BRANCH = "main"          # or your own branch, e.g. "dev_mahir"
DATASET = "fsc"          # "fsc" or "l2arctic"
EPOCHS = 15
NOISE = "white"          # "white" or "none"
UNTRAINED = False        # True only for the untrained baseline (T18)
SPLIT = "all"            # "all" = random split + 5 folds; or "random", "folds", "fold1" ...
DRIVE = "/content/drive/MyDrive/TenVoices"   # the team folder in your Drive

## 2. Code, packages and data

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%cd /content
!git clone -q https://github.com/minhaz-42/Voices-Speech-Recognition-445.git repo || true
%cd /content/repo
!git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
# Keep Colab's own CUDA build of torch; install everything else at the pinned versions.
!grep -v '^torch==' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt

In [ ]:
import os, subprocess
raw = {"fsc": f"{DRIVE}/raw/fsc/fluent.tar.gz", "l2arctic": f"{DRIVE}/raw/l2arctic"}[DATASET]
os.makedirs(f"data/raw/{DATASET}", exist_ok=True)
if DATASET == "fsc" and not os.path.isdir("data/raw/fsc/fluent_speech_commands_dataset"):
    subprocess.run(["tar", "-xzf", raw, "-C", "data/raw/fsc"], check=True)
elif DATASET == "l2arctic":
    subprocess.run(f"cp -rn '{raw}'/* data/raw/l2arctic/", shell=True, check=True)
!python -m support.dataset --dataset {DATASET} --build

## 3. Train and evaluate

In [ ]:
flags = "--untrained" if UNTRAINED else f"--epochs {EPOCHS} --noise {NOISE}"
!python -m support.train_wav2vec2 --dataset {DATASET} --split {SPLIT} {flags}

## 4. Keep the results

- **Models** (large, never in git) are copied to `TenVoices/models/` in the Drive.
- **Results** (small CSV / JSON) are zipped to the Drive. Unzip them into `results/` of your local repository, then commit and push them from your own branch as usual.

In [ ]:
from support.train_wav2vec2 import default_run_name
run = default_run_name(EPOCHS, NOISE, UNTRAINED)
!mkdir -p "{DRIVE}/models" "{DRIVE}/results"
!cp -r outputs/models/{run} "{DRIVE}/models/" 2>/dev/null || echo "no saved models"
!cd results && zip -qr "{DRIVE}/results/{run}_{DATASET}.zip" {run}/{DATASET} && echo "saved {DRIVE}/results/{run}_{DATASET}.zip"